In [32]:
from __future__ import annotations

import os
import re
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import scipy.io as sio
import scipy.signal as sps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [33]:
# =========================
# Config
# =========================
@dataclass
class CFG:
    # ---- data ----
    PATIENT_DIR: Path = Path(r"D:\SWEC ETHZ dataset\_extracted\ID1")
    FS_DEFAULT: float = 512.0

    # Define pseudo pre/post segments (your existing labeling)
    PRE_POST_SEC: float = 180.0  # 3 minutes

    # ---- preprocessing ----
    NOTCH_FREQ: float = 50.0
    NOTCH_Q: float = 30.0
    BANDPASS: Optional[Tuple[float, float]] = None  # e.g. (0.5, 150.0) or None
    USE_CAR: bool = True

    # Z-score normalization + z-score-based artifact handling
    USE_ZSCORE: bool = True
    RMS_ZTHRESH: float = 5.0          # RMS-z threshold for "bad" samples
    CLIP_MAD_K: float = 3.0           # clip to median ± k*MAD at bad time points

    # ---- windowing ----
    WIN_SEC: float = 2.0

    # training: variable hop to oversample ictal
    HOP_PRE_SEC: float = 1.0
    HOP_ICTAL_SEC: float = 0.25
    HOP_POST_SEC: float = 1.0

    # testing: fixed hop
    HOP_TEST_SEC: float = 1.0

    # boundary handling (train only)
    GUARD_SEC: float = 1.0

    # window-level artifact reject (keep from your pipeline; optional)
    USE_WIN_AMP_REJECT: bool = True
    AMP_THRESH: float = 500.0  # set to your AMP_UV_THRESH (units depend on dataset)

    # ---- STFT ----
    N_FFT: int = 256
    STFT_HOP: int = 64
    STFT_WIN: int = 256
    FMIN: float = 1.0
    FMAX: float = 120.0

    # ---- training ----
    SEED: int = 42
    BATCH_SIZE: int = 64
    NUM_WORKERS: int = 0
    LR: float = 1e-3
    WD: float = 1e-4
    EPOCHS: int = 40
    PATIENCE: int = 7
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"


LABELS = {"preictal": 0, "ictal": 1, "postictal": 2}


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [34]:
# =========================
# Load SWEC .mat (your way)
# =========================
def load_swec_mat(mat_path: Path, fs_default: float) -> Tuple[np.ndarray, float]:
    mat = scipy.io.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    if "EEG" not in mat:
        raise KeyError(f"{mat_path.name}: missing EEG")

    eeg = np.asarray(mat["EEG"])
    if eeg.ndim != 2:
        raise ValueError(f"{mat_path.name}: EEG has shape {eeg.shape}")

    if eeg.shape[0] > eeg.shape[1]:
        eeg = eeg.T
    if eeg.shape[0] > eeg.shape[1]:
        eeg = eeg.T

    fs = fs_default
    for k in ["fs", "Fs", "srate", "sampling_rate"]:
        if k in mat:
            try:
                fs = float(np.asarray(mat[k]).item())
                break
            except Exception:
                pass

    return eeg.astype(np.float32), float(fs)

In [35]:
# =========================
# Preprocessing
# =========================
def notch_filter(x: np.ndarray, fs: float, f0: float, q: float) -> np.ndarray:
    b, a = sps.iirnotch(w0=f0, Q=q, fs=fs)
    return sps.filtfilt(b, a, x, axis=1).astype(np.float32)


def bandpass_filter(x: np.ndarray, fs: float, lo: float, hi: float, order: int = 4) -> np.ndarray:
    b, a = sps.butter(order, [lo, hi], btype="bandpass", fs=fs)
    return sps.filtfilt(b, a, x, axis=1).astype(np.float32)


def common_average_reference(x: np.ndarray) -> np.ndarray:
    return (x - x.mean(axis=0, keepdims=True)).astype(np.float32)


def zscore_per_channel(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    mu = x.mean(axis=1, keepdims=True)
    sd = x.std(axis=1, keepdims=True)
    return ((x - mu) / (sd + eps)).astype(np.float32)


def artifact_rms_z_clip(x: np.ndarray, zthresh: float, clip_mad_k: float) -> np.ndarray:
    """
    Artifact handling:
      - RMS across channels per time sample
      - z-score RMS
      - samples where RMS-z > zthresh are clipped channel-wise to median ± k*MAD
    """
    rms = np.sqrt((x ** 2).mean(axis=0))
    rz = (rms - rms.mean()) / (rms.std() + 1e-8)
    bad = rz > zthresh
    if not bad.any():
        return x.astype(np.float32)

    med = np.median(x, axis=1, keepdims=True)
    mad = np.median(np.abs(x - med), axis=1, keepdims=True) + 1e-8
    lo = med - clip_mad_k * mad
    hi = med + clip_mad_k * mad

    x2 = x.copy()
    x2[:, bad] = np.clip(x2[:, bad], lo, hi)
    return x2.astype(np.float32)


def preprocess(x: np.ndarray, fs: float, cfg: CFG) -> np.ndarray:
    x = notch_filter(x, fs, cfg.NOTCH_FREQ, cfg.NOTCH_Q)
    if cfg.BANDPASS is not None:
        x = bandpass_filter(x, fs, cfg.BANDPASS[0], cfg.BANDPASS[1], order=4)
    if cfg.USE_CAR:
        x = common_average_reference(x)
    if cfg.USE_ZSCORE:
        x = zscore_per_channel(x)
        x = artifact_rms_z_clip(x, zthresh=cfg.RMS_ZTHRESH, clip_mad_k=cfg.CLIP_MAD_K)
    return x.astype(np.float32)



In [36]:
# =========================
# Windowing (3-class from pseudo start/end)
# =========================
def label_by_center(center: int, start_idx: int, end_idx: int) -> int:
    if center < start_idx:
        return LABELS["preictal"]
    elif center <= end_idx:
        return LABELS["ictal"]
    else:
        return LABELS["postictal"]


def make_windows_3class(
    n_samples: int,
    fs: float,
    start_idx: int,
    end_idx: int,
    cfg: CFG,
    training: bool
) -> List[Tuple[int, int, int]]:
    win = int(round(cfg.WIN_SEC * fs))
    guard = int(round(cfg.GUARD_SEC * fs))

    out: List[Tuple[int, int, int]] = []

    if not training:
        hop = int(round(cfg.HOP_TEST_SEC * fs))
        for s in range(0, n_samples - win + 1, hop):
            e = s + win
            c = (s + e) // 2
            y = label_by_center(c, start_idx, end_idx)
            out.append((s, e, y))
        return out

    ptr = 0
    while ptr <= n_samples - win:
        s = ptr
        e = s + win
        c = (s + e) // 2
        y = label_by_center(c, start_idx, end_idx)

        # boundary drop (train only)
        if not (abs(c - start_idx) <= guard or abs(c - end_idx) <= guard):
            out.append((s, e, y))

        # variable hop
        if y == LABELS["ictal"]:
            hop_sec = cfg.HOP_ICTAL_SEC
        elif y == LABELS["preictal"]:
            hop_sec = cfg.HOP_PRE_SEC
        else:
            hop_sec = cfg.HOP_POST_SEC

        hop = max(1, int(round(hop_sec * fs)))
        ptr += hop

    return out

In [37]:
# =========================
# STFT features (pool over channels)
# =========================
def stft_logpower_pool(x_win: torch.Tensor, fs: float, cfg: CFG) -> torch.Tensor:
    # x_win: [C, N]
    window = torch.hann_window(cfg.STFT_WIN, device=x_win.device)
    X = torch.stft(
        x_win,
        n_fft=cfg.N_FFT,
        hop_length=cfg.STFT_HOP,
        win_length=cfg.STFT_WIN,
        window=window,
        center=False,
        return_complex=True,
    )  # [C, F, T]

    P = torch.log1p(X.real ** 2 + X.imag ** 2)

    freqs = torch.fft.rfftfreq(cfg.N_FFT, d=1.0 / fs).to(P.device)
    mask = (freqs >= cfg.FMIN) & (freqs <= cfg.FMAX)
    P = P[:, mask, :]  # [C, F', T]

    mean_map = P.mean(dim=0)
    std_map = P.std(dim=0)
    max_map = P.max(dim=0).values
    return torch.stack([mean_map, std_map, max_map], dim=0)  # [3, F', T]

In [38]:
# =========================
# Dataset
# =========================
class WindowDS(Dataset):
    def __init__(self, runs: List[Dict], cfg: CFG, training: bool):
        self.runs = runs
        self.cfg = cfg
        self.training = training
        self.index: List[Tuple[int, int, int, int]] = []

        for i, r in enumerate(runs):
            n = r["x"].shape[1]
            windows = make_windows_3class(
                n_samples=n,
                fs=r["fs"],
                start_idx=r["start_idx"],
                end_idx=r["end_idx"],
                cfg=cfg,
                training=training
            )

            for (s, e, y) in windows:
                if cfg.USE_WIN_AMP_REJECT:
                    seg = r["x"][:, s:e]
                    if np.max(np.abs(seg)) > cfg.AMP_THRESH:
                        continue
                self.index.append((i, s, e, y))

        if not self.index:
            raise RuntimeError("No windows after filtering. Check AMP_THRESH / preprocessing settings.")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx: int):
        run_i, s, e, y = self.index[idx]
        r = self.runs[run_i]
        seg = r["x"][:, s:e]  # [C, N]
        feat = stft_logpower_pool(torch.from_numpy(seg), r["fs"], self.cfg)
        return feat, torch.tensor(y, dtype=torch.long)


In [39]:
# =========================
# Model
# =========================
class Small2DCNN(nn.Module):
    def __init__(self, n_classes: int = 3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Linear(64, n_classes)

    def forward(self, x):
        h = self.net(x).squeeze(-1).squeeze(-1)
        return self.fc(h)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: str):
    model.eval()
    ys, yh = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        pred = model(xb).argmax(dim=1)
        ys.append(yb.cpu().numpy())
        yh.append(pred.cpu().numpy())
    ys = np.concatenate(ys)
    yh = np.concatenate(yh)
    acc = accuracy_score(ys, yh)
    f1m = f1_score(ys, yh, average="macro")
    cm = confusion_matrix(ys, yh, labels=[0, 1, 2])
    return acc, f1m, cm



In [40]:
# =========================
# Training: Leave-one-seizure-out on ONE patient
# =========================
def train_loso_one_patient(cfg: CFG, runs: List[Dict]):
    device = cfg.DEVICE
    scores = []

    for test_i in range(len(runs)):
        train_runs = [r for j, r in enumerate(runs) if j != test_i]
        test_runs = [runs[test_i]]

        train_ds = WindowDS(train_runs, cfg, training=True)
        test_ds = WindowDS(test_runs, cfg, training=False)

        y_train = np.array([y for (_, _, _, y) in train_ds.index], dtype=np.int64)
        counts = np.bincount(y_train, minlength=3)
        class_w = 1.0 / np.maximum(counts, 1)
        sample_w = class_w[y_train]

        sampler = WeightedRandomSampler(
            weights=torch.from_numpy(sample_w).double(),
            num_samples=len(sample_w),
            replacement=True
        )

        train_loader = DataLoader(
            train_ds,
            batch_size=cfg.BATCH_SIZE,
            sampler=sampler,
            num_workers=cfg.NUM_WORKERS,
            pin_memory=True
        )
        test_loader = DataLoader(
            test_ds,
            batch_size=cfg.BATCH_SIZE,
            shuffle=False,
            num_workers=cfg.NUM_WORKERS,
            pin_memory=True
        )

        model = Small2DCNN(n_classes=3).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WD)
        crit = nn.CrossEntropyLoss(weight=torch.tensor(class_w, dtype=torch.float32, device=device))

        best_f1 = -1.0
        best_state = None
        bad = 0

        print("\n" + "=" * 70)
        print(f"[Fold {test_i + 1}/{len(runs)}] TEST={runs[test_i]['name']}")
        print(f"Train windows={len(train_ds)} | Test windows={len(test_ds)} | class_counts(train)={counts.tolist()}")

        for epoch in range(1, cfg.EPOCHS + 1):
            model.train()
            loss_sum = 0.0
            nb = 0

            for xb, yb in train_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                opt.zero_grad(set_to_none=True)
                logits = model(xb)
                loss = crit(logits, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

                loss_sum += float(loss.item())
                nb += 1

            acc, f1m, _ = evaluate(model, test_loader, device)
            print(f"  epoch {epoch:02d} | loss={loss_sum / max(1, nb):.4f} | test_acc={acc:.4f} | test_f1m={f1m:.4f}")

            if f1m > best_f1:
                best_f1 = f1m
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad = 0
            else:
                bad += 1
                if bad >= cfg.PATIENCE:
                    break

        if best_state is not None:
            model.load_state_dict(best_state)

        acc, f1m, cm = evaluate(model, test_loader, device)
        print("[BEST] acc:", acc, "macro-F1:", f1m)
        print("CM rows=true cols=pred [pre, ictal, post]\n", cm)

        scores.append((acc, f1m))

    print("\n" + "=" * 70)
    print("FINAL mean acc:", float(np.mean([a for a, _ in scores])))
    print("FINAL mean macro-F1:", float(np.mean([f for _, f in scores])))


In [41]:
# =========================
# Main
# =========================
def main():
    cfg = CFG()
    set_seed(cfg.SEED)

    mats = sorted(cfg.PATIENT_DIR.glob("Sz*.mat"), key=lambda p: int(re.findall(r"\d+", p.stem)[0]))
    if not mats:
        raise FileNotFoundError(f"No Sz*.mat in {cfg.PATIENT_DIR}")

    runs = []
    for p in mats:
        x, fs = load_swec_mat(p, cfg.FS_DEFAULT)

        n_ch, n_samp = x.shape
        pre = int(cfg.PRE_POST_SEC * fs)
        start_idx = pre
        end_idx = n_samp - pre
        if end_idx <= start_idx:
            raise ValueError(f"{p.name}: too short for {cfg.PRE_POST_SEC}s pre/post")

        # preprocess
        x = preprocess(x, fs, cfg)

        runs.append({
            "name": p.name,
            "x": x,
            "fs": fs,
            "start_idx": int(start_idx),
            "end_idx": int(end_idx),
        })

        print(f"[load] {p.name} shape={x.shape} fs={fs} start_idx={start_idx} end_idx={end_idx}")

    train_loso_one_patient(cfg, runs)


if __name__ == "__main__":
    main()

[load] Sz1.mat shape=(47, 248320) fs=512.0 start_idx=92160 end_idx=156160
[load] Sz2.mat shape=(47, 220160) fs=512.0 start_idx=92160 end_idx=128000
[load] Sz3.mat shape=(47, 253952) fs=512.0 start_idx=92160 end_idx=161792
[load] Sz4.mat shape=(47, 189440) fs=512.0 start_idx=92160 end_idx=97280
[load] Sz5.mat shape=(47, 313344) fs=512.0 start_idx=92160 end_idx=221184
[load] Sz6.mat shape=(47, 214528) fs=512.0 start_idx=92160 end_idx=122368
[load] Sz7.mat shape=(47, 205824) fs=512.0 start_idx=92160 end_idx=113664
[load] Sz8.mat shape=(47, 218112) fs=512.0 start_idx=92160 end_idx=125952
[load] Sz9.mat shape=(47, 189952) fs=512.0 start_idx=92160 end_idx=97792
[load] Sz10.mat shape=(47, 190464) fs=512.0 start_idx=92160 end_idx=98304
[load] Sz11.mat shape=(47, 194560) fs=512.0 start_idx=92160 end_idx=102400
[load] Sz12.mat shape=(47, 221696) fs=512.0 start_idx=92160 end_idx=129536
[load] Sz13.mat shape=(47, 208896) fs=512.0 start_idx=92160 end_idx=116736

[Fold 1/13] TEST=Sz1.mat
Train windo